In [ ]:
!pip install -q transformers bitsandbytes accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Using Unsloth's public 4-bit quantization of Llama 3.1 8B
model_id = "unsloth/llama-3.1-8b-Instruct-bnb-4bit"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model (this takes about 1-2 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("Llama 3.1 8B is ready!")

# Define your prompt using Llama 3.1's specific chat template structure
messages = [
    {"role": "system", "content": "You are a helpful and concise AI assistant."},
    {"role": "user", "content": "Explain the concept of 'Technical Debt' to a non-technical person using a cooking analogy."}
]

# Format the template
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Generate response
inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
print("\nGenerating response...\n")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.6,
    do_sample=True
)

# Decode and print output
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, outputs)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

Loading tokenizer...


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

#SINHALA LANGUAGE

In [ ]:
import json

DATA_PATH = "/content/sample_data/sri_lankan_simplified.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} items")

Loaded 203 items


In [ ]:
INSTRUCTION_PROMPT = (
    """ You are a Sinhala Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Sri Lankan context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.

Question: {question}
Option A: {option A}
Option B: {option B}
Option C: පිළිතුරු දෙකම නිවැරදියි.
Option D: පිළිතුරු දෙකම නිවැරදි නොවේ.
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.
"""
)

def build_prompt(item):
    lines = [INSTRUCTION_PROMPT, "", item["scenario_question"], "", "Options:"]
    for letter, text in item["options"].items():
        lines.append(f"{letter}. {text}")
    return "\n".join(lines)

In [ ]:
import re
def generate_answer(prompt, valid_letters, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_len = inputs["input_ids"].shape[-1]
    raw = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    # First try a clean letter match (model followed instructions)
    match = re.search(r"\b([A-D])\b", raw.upper())
    if match and match.group(1) in valid_letters:
      predicted = match.group(1)
    # Fall back: model echoed the option content instead of the letter
    elif "D" in raw:
      predicted = "0"
    elif "C" in raw:   # check this AFTER the "නොවේ" (negative) case
      predicted = "Both"
    #else:
      #  predicted = None

    return predicted, raw

In [ ]:
OUTPUT_PATH = "/content/sri_lankan_results.json"

for i, item in enumerate(data):
    if item.get("model_answer"):  # already processed, skip (resume-safe)
        continue

    prompt = build_prompt(item)
    predicted, raw_output = generate_answer(prompt, list(item["options"].keys()))

    item["model_answer"] = predicted
    item["raw_model_output"] = raw_output

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"[{i+1}/{len(data)}] {item['id']} -> {predicted} (gold: {item['gold_answer']})")

print("All done.")

[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1/203] SI_2004 -> A (gold: A)


[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/203] SI_2007 -> Both (gold: B)


[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[200/203] SI_2980 -> A (gold: 0)


[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[201/203] SI_2990 -> Both (gold: 0)


[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[202/203] SI_2992 -> Both (gold: 0)
[203/203] SI_2993 -> Both (gold: 0)
All done.


In [ ]:
import json

with open("/content/sri_lankan_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

correct = sum(1 for item in results if item.get("model_answer") == item.get("gold_answer"))
total = len(results)

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {correct / total:.2%}")

Correct: 71 / 203
Accuracy: 34.98%


#CHINESE LANGUAGE

In [ ]:
import json

DATA_PATH = "/content/sample_data/chinese_simplified.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} items")

Loaded 790 items


In [ ]:
INSTRUCTION_PROMPT = (
    """
You are a Chinese Expert for answering multiple-choice questions.
You need to always evaluate these questions based on the Chinese context and provide the most accurate answer.
You should only respond with the correct option text, without any additional explanation or commentary.

Question: {question}
Option A: {option A}
Option B: {option B}
Option C: {option C}
Option D: {option D}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'. "Question": "

"""
)

def build_prompt(item):
    lines = [INSTRUCTION_PROMPT, "", item["scenario_question"], "", "Options:"]
    for letter, text in item["options"].items():
        lines.append(f"{letter}. {text}")
    return "\n".join(lines)

In [ ]:
import re
def generate_answer(prompt, valid_letters, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_len = inputs["input_ids"].shape[-1]
    raw = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    # First try a clean letter match (model followed instructions)
    match = re.search(r"\b([A-D])\b", raw.upper())
    if match and match.group(1) in valid_letters:
        predicted = match.group(1)
    else:
        predicted = None

    return predicted, raw

In [ ]:
OUTPUT_PATH = "/content/chinese_results.json"

for i, item in enumerate(data):
    if item.get("model_answer"):  # already processed, skip (resume-safe)
        continue

    prompt = build_prompt(item)
    predicted, raw_output = generate_answer(prompt, list(item["options"].keys()))

    item["model_answer"] = predicted
    item["raw_model_output"] = raw_output

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"[{i+1}/{len(data)}] {item['id']} -> {predicted} (gold: {item['gold_answer']})")

print("All done.")

[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1/790] ZH_0_0 -> B (gold: A)


[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/790] ZH_0_3 -> A (gold: C)


[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[787/790] ZH_157_35 -> C (gold: C)


[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[788/790] ZH_157_40 -> A (gold: D)


[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[789/790] ZH_157_65 -> A (gold: A)
[790/790] ZH_157_75 -> A (gold: D)
All done.


In [ ]:
import json

with open("/content/chinese_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

correct = sum(1 for item in results if item.get("model_answer") == item.get("gold_answer"))
total = len(results)

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {correct / total:.2%}")

Correct: 232 / 790
Accuracy: 29.37%


#INDONESIAN LANGUAGE

In [ ]:
import json

DATA_PATH = "/content/sample_data/indonesian_simplified.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} items")

Loaded 366 items


In [ ]:
INSTRUCTION_PROMPT = (
    """
    You are an Indonesian Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Indonesian context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.

Question: {question}
Option A: {option A}
Option B: {option B}
Option C: {option C}
Option D: {option D}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.


"""
)

def build_prompt(item):
    lines = [INSTRUCTION_PROMPT, "", item["scenario_question"], "", "Options:"]
    for letter, text in item["options"].items():
        lines.append(f"{letter}. {text}")
    return "\n".join(lines)

In [ ]:
import re
def generate_answer(prompt, valid_letters, max_new_tokens=20):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    input_len = inputs["input_ids"].shape[-1]
    raw = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    # First try a clean letter match (model followed instructions)
    match = re.search(r"\b([A-D])\b", raw.upper())
    if match and match.group(1) in valid_letters:
        predicted = match.group(1)
    else:
        predicted = None

    return predicted, raw

In [ ]:
OUTPUT_PATH = "/content/indonesian_results.json"

for i, item in enumerate(data):
    if item.get("model_answer"):  # already processed, skip (resume-safe)
        continue

    prompt = build_prompt(item)
    predicted, raw_output = generate_answer(prompt, list(item["options"].keys()))

    item["model_answer"] = predicted
    item["raw_model_output"] = raw_output

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"[{i+1}/{len(data)}] {item['id']} -> {predicted} (gold: {item['gold_answer']})")

print("All done.")

[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1/366] ID_3 -> A (gold: A)


[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/366] ID_5 -> B (gold: B)


[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[363/366] ID_2019 -> B (gold: B)


[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[364/366] ID_2040 -> B (gold: B)


[transformers] Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[365/366] ID_2050 -> C (gold: C)
[366/366] ID_2058 -> C (gold: C, D)
All done.


In [ ]:
import json

with open("/content/indonesian_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

correct = 0
for item in results:
    model_answer = item.get("model_answer")
    gold_answer = item.get("gold_answer", "")

    gold_letters = [g.strip() for g in gold_answer.split(",")]  # handles single or multiple gold answers

    if model_answer is not None and model_answer.strip() in gold_letters:
        correct += 1

total = len(results)

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {correct / total:.2%}")

Correct: 209 / 366
Accuracy: 57.10%
